## 01 — Using Tippecanoe

`tippecanoe` is a command-line tool from Mapbox that converts GeoJSON into a vector tile pyramid.

One command replaces our entire Module 02 pipeline — and produces a smaller, faster output format. This notebook runs it, inspects the output, and maps every flag to a decision we already made by hand.

## Installation

On macOS with Homebrew:

```bash
brew install tippecanoe
```

On Linux (Ubuntu/Debian):

```bash
sudo apt-get install tippecanoe
```

Verify the install:

In [2]:
import subprocess
result = subprocess.run(["tippecanoe", "--version"], capture_output=True, text=True)
print(result.stdout or result.stderr)

FileNotFoundError: [WinError 2] The system cannot find the file specified

## Running Tippecanoe

The basic command:

```bash
tippecanoe \
  --output=railroads.pmtiles \
  --minimum-zoom=1 \
  --maximum-zoom=14 \
  --simplification=10 \
  --drop-densest-as-needed \
  --layer=railroads \
  ne_10m_railroads.geojson
```

Let's run it from Python and capture the output:

In [ ]:
from pathlib import Path
import subprocess
import time

input_file  = Path("../../data/ne_10m_railroads.geojson")
output_file = Path("../../data/railroads.pmtiles")

cmd = [
    "tippecanoe",
    f"--output={output_file}",
    "--force",                     # overwrite if exists
    "--minimum-zoom=1",
    "--maximum-zoom=14",
    "--simplification=10",         # Douglas-Peucker tolerance in tile pixels
    "--drop-densest-as-needed",    # drop features at low zoom if tile is too large
    "--layer=railroads",
    str(input_file),
]

t0 = time.perf_counter()
result = subprocess.run(cmd, capture_output=True, text=True)
elapsed = time.perf_counter() - t0

print(result.stderr)   # tippecanoe writes progress to stderr
print(f"\nCompleted in {elapsed:.1f}s")

## Inspecting the Output

In [ ]:
size_mb = output_file.stat().st_size / 1_000_000
raw_mb  = input_file.stat().st_size  / 1_000_000

print(f"Input  (raw GeoJSON):  {raw_mb:.1f} MB")
print(f"Output (PMTiles):      {size_mb:.2f} MB")
print(f"Compression ratio:     {raw_mb / size_mb:.1f}×  smaller")

## Mapping Flags to Decisions We Already Made

Every `tippecanoe` flag corresponds to something we built or decided manually:

| tippecanoe flag | What it does | Our equivalent |
|-----------------|-------------|----------------|
| `--minimum-zoom` | First zoom level that gets tiles | Bottom of our LOD range |
| `--maximum-zoom` | Most detailed zoom level | Top of our LOD range |
| `--simplification=10` | D-P tolerance in tile pixels per zoom | Our epsilon per LOD level |
| `--drop-densest-as-needed` | Remove least-important features when tile is too large | Our `scalerank <= 4` coarse filter |
| `--layer=railroads` | Names the data layer in the tile | Our filename convention |

The flags we do NOT have to specify:
- Viewport culling — built into the tile addressing scheme
- Binary encoding — automatic (MVT format)
- Tile pyramid structure — automatic
- Spatial index — automatic (tiles ARE the index)
- Zoom-driven switching — automatic (client requests the right `{z}` tiles)


## Inspecting Tile Contents with sqlite3

PMTiles can be converted to `.mbtiles` (SQLite) for inspection. Or we can use the `pmtiles` CLI to peek at specific tiles.

Alternatively, inspect the metadata embedded in the PMTiles file:

In [ ]:
# Use tippecanoe's companion tool to show metadata
result = subprocess.run(
    ["tile-join", "--no-tile-compression", "--if-matched",
     f"--output={output_file.with_suffix('.inspect.pmtiles')}",
     str(output_file)],
    capture_output=True, text=True
)

# Simpler: just show what pmtiles show gives us
result2 = subprocess.run(
    ["pmtiles", "show", str(output_file)],
    capture_output=True, text=True
)
print(result2.stdout or result2.stderr or "(install pmtiles CLI: pip install pmtiles)")

## Viewing in ipyleaflet

ipyleaflet supports PMTiles through the `PMTilesLayer` (requires `ipyleaflet >= 0.18`).

For local files, we need to serve them via a local HTTP server or use `localtileserver`.

In [ ]:
# Try loading with localtileserver if available
try:
    from localtileserver import TileClient, get_leaflet_tile_layer
    from ipyleaflet import Map

    client = TileClient(str(output_file))
    layer  = get_leaflet_tile_layer(client)
    m = Map(center=client.center(), zoom=client.default_zoom)
    m.add(layer)
    m
except ImportError:
    print("localtileserver not installed.")
    print("Install with: pip install localtileserver")
    print()
    print("Alternative: upload railroads.pmtiles to https://pmtiles.io to view it online.")
    print(f"File location: {output_file.resolve()}")

## Exercise A

Run `tippecanoe` a second time with `--maximum-zoom=8` and compare the output file size.

Then answer: what did limiting the maximum zoom cost us in terms of user experience, and what did it save?

In [3]:
output_z8 = Path("../../data/railroads_z8.pmtiles")

cmd_z8 = [
    "tippecanoe",
    f"--output={output_z8}",
    "--force",
    "--minimum-zoom=1",
    "--maximum-zoom=8",
    "--simplification=10",
    "--drop-densest-as-needed",
    "--layer=railroads",
    str(input_file),
]

t0 = time.perf_counter()
result_z8 = subprocess.run(cmd_z8, capture_output=True, text=True)
elapsed_z8 = time.perf_counter() - t0

print(result_z8.stderr)
print(f"Completed in {elapsed_z8:.1f}s\n")

if output_z8.exists() and output_file.exists():
    size_z14 = output_file.stat().st_size  / 1_000_000
    size_z8  = output_z8.stat().st_size    / 1_000_000
    print(f"zoom 1-14:  {size_z14:.2f} MB")
    print(f"zoom 1-8:   {size_z8:.2f}  MB")
    print(f"Saving:     {size_z14 - size_z8:.2f} MB  ({(1 - size_z8/size_z14)*100:.0f}% smaller)")

# What limiting to zoom 8 costs and saves:
#
# COST:
#   At zoom 9 and above (city district and street level), the map shows no railroad
#   data from this layer at all. A user zooming into a specific station or yard to
#   see fine track layout would see nothing. The maximum useful zoom is roughly a
#   full city view (~30km wide), which is appropriate for a network-level map but
#   not for local navigation.
#
# SAVING:
#   Zoom levels 9-14 account for the vast majority of tiles in a full pyramid
#   because the number of tiles quadruples with each zoom level (2^z * 2^z).
#   Zoom 14 alone has 2^14 x 2^14 = 268 million possible tiles — more than all
#   lower zoom levels combined. Capping at zoom 8 typically reduces file size by
#   60-80% and cuts tippecanoe build time proportionally.

NameError: name 'Path' is not defined

## Exercise B

The `--simplification=10` flag sets the tolerance in **tile pixels**, not degrees. At zoom 14, a tile covers roughly 2.4km × 2.4km in 4096 pixels — so one pixel ≈ 0.6m.

Calculate what `--simplification=10` means in meters at zoom levels 2, 5, 8, and 12. Compare these to the degree-based epsilon values we chose in Module 02.

In [ ]:
import math

EARTH_CIRC_M = 2 * math.pi * 6_378_137
TILE_PIXELS  = 4096
SIMPLIFY_PX  = 10

print(f"{'Zoom':>6}  {'Tile width (km)':>16}  {'1 pixel (m)':>12}  {'10-px tolerance (m)':>22}")
print("-" * 62)
for z in [2, 5, 8, 12]:
    tile_m = EARTH_CIRC_M / (2 ** z)
    px_m   = tile_m / TILE_PIXELS
    tol_m  = SIMPLIFY_PX * px_m
    print(f"{z:>6}  {tile_m/1000:>16.1f}  {px_m:>12.2f}  {tol_m:>22.1f}")

deg_to_m = 111_000
print()
print(f"{'LOD level':<12}  {'epsilon (°)':>12}  {'epsilon (m)':>12}  {'zoom range':>12}")
print("-" * 52)
for name, (eps, zoom_max) in [
        ("coarse",     (1.0,    4)),
        ("medium",     (0.1,    6)),
        ("fine",       (0.01,  10)),
        ("extra_fine", (0.001, 14)),
]:
    print(f"{name:<12}  {eps:>12}  {eps * deg_to_m:>12,.0f}  up to zoom {zoom_max:>2}")

# Comparison:
# Tippecanoe's per-pixel tolerance scales continuously with zoom, so the
# simplification is always proportional to what a human eye can see on screen:
#
#   zoom  2: ~24,460 m  vs our coarse     ~111,000 m  — tippecanoe 4.5x tighter
#   zoom  5:  ~3,058 m  vs our medium      ~11,100 m  — tippecanoe 3.6x tighter
#   zoom  8:    ~382 m  vs our fine         ~1,110 m  — tippecanoe 2.9x tighter
#   zoom 12:     ~24 m  vs our extra_fine     ~111 m  — tippecanoe 4.6x tighter
#
# Our fixed epsilons were chosen conservatively in degrees without accounting for
# zoom level, so they preserve more detail than necessary at every level. Tippecanoe
# removes exactly as much as is invisible at each zoom, no more, no less.
# The result is smaller tiles without any visible quality difference.

## Check Your Understanding

We ran `tippecanoe` with `--drop-densest-as-needed`. This flag tells tippecanoe to automatically drop the least-important features when a tile would otherwise be too large.

How does tippecanoe decide which features are "least important"? And how does that compare to our manual `scalerank <= 4` filter? Which approach is more principled — and what are the tradeoffs of each?

---

In [ ]:
# How tippecanoe decides which features are "least important" with --drop-densest-as-needed:
#
# Tippecanoe measures tile size during generation. If a tile would exceed its size
# limit, it sorts features by density (how many features occupy the same area) and
# drops the densest clusters first, keeping spatially isolated features. It does
# NOT use an explicit importance attribute like scalerank unless you configure it
# to do so with a --feature-filter or --use-attribute-for-id flag. The default
# heuristic is purely spatial. Features that are crowded together are deemed
# redundant and removed.
#
# Our scalerank <= 4 filter, by contrast, is an explicit semantic ranking. We
# kept only lines that Natural Earth editors classified as globally significant
# (main trunk lines, major intercity routes). A line that happens to pass through
# a dense region is kept, a minor spur in a sparse region is dropped.
#
# Our scalerank filter is more principled for what to show at low zoom.
# It preserves the structurally important network skeleton regardless of where
# features happen to cluster on screen. Tippecanoe's density-based drop is more
# principled for controlling tile size as it removes the minimum data needed to
# hit a size target without caring about semantic importance.
#
# Tradeoff: scalerank filtering can leave holes in sparse regions (e.g., Africa)
# where even minor lines would be the only visible railroad. Tippecanoe's approach
# can drop important lines in dense regions (central Europe) simply because there
# are many of them, resulting in an arbitrarily thinned network rather than a
# semantically meaningful one. Combining both, prefiltering by scalerank AND
# using tippecanoe's size management, is the production-quality approach.

## Next

In [02 — The Comparison](./02-The_Comparison.ipynb), we put both systems side by side and answer the final question: what did `tippecanoe` actually save us from?